# Spatial Canvas Analytics

This notebook visualises **Zylvex Spatial Canvas anchor data** — digital content anchored to physical locations via AR.

**What you'll see:**
- Interactive Folium map with clustered anchor markers, coloured by category
- Reaction heatmap overlay showing engagement hotspots
- Bar chart: anchor density by city/region
- Time-series: anchor creation rate over 90 days
- 3D scatter: latitude × longitude × reaction count (coloured by category)
- Interactive **Sandbox** section to filter by category, reactions, and date range

A sample dataset of 100 anchors is generated with NumPy — no external files required.

In [ ]:
!pip install numpy plotly folium ipywidgets --quiet

In [ ]:
import warnings
import datetime
import numpy as np
import plotly.graph_objects as go
import folium
from folium.plugins import MarkerCluster, HeatMap
import ipywidgets as widgets
from IPython.display import display, IFrame

warnings.filterwarnings('ignore')
rng = np.random.default_rng(42)

In [ ]:
# Generate 100 sample anchors
CITIES = {
    'New York':  (40.7128, -74.0060),
    'Los Angeles': (34.0522, -118.2437),
    'London':    (51.5074, -0.1278),
    'Tokyo':     (35.6762, 139.6503),
}
CATEGORIES = ['art', 'food', 'tech', 'nature', 'history']
CAT_COLORS = {
    'art': 'red',
    'food': 'orange',
    'tech': 'blue',
    'nature': 'green',
    'history': 'purple',
}
CAT_HEX = {
    'art': '#ef4444',
    'food': '#f97316',
    'tech': '#6366f1',
    'nature': '#10b981',
    'history': '#8b5cf6',
}

N = 100
city_names = list(CITIES.keys())
city_assignments = rng.choice(city_names, size=N)

lats, lngs = [], []
for city in city_assignments:
    base_lat, base_lng = CITIES[city]
    lats.append(base_lat + rng.normal(0, 0.08))
    lngs.append(base_lng + rng.normal(0, 0.08))

lats = np.array(lats)
lngs = np.array(lngs)
categories = rng.choice(CATEGORIES, size=N)
reaction_counts = rng.integers(0, 151, size=N)

# Use a fixed reference date for reproducibility
base_date = datetime.date(2024, 10, 1)
day_offsets = rng.integers(0, 90, size=N)
created_dates = [base_date + datetime.timedelta(days=int(d)) for d in day_offsets]

anchors = [
    {
        'id': i,
        'title': f'{cat.capitalize()} Anchor #{i}',
        'lat': float(lats[i]),
        'lng': float(lngs[i]),
        'category': cat,
        'reaction_count': int(reaction_counts[i]),
        'city': city_assignments[i],
        'created_at': created_dates[i],
    }
    for i, cat in enumerate(categories)
]

print(f'Generated {len(anchors)} anchors')
print(f'Category distribution: { {c: int((categories == c).sum()) for c in CATEGORIES} }')
print(f'City distribution: { {c: int((city_assignments == c).sum()) for c in city_names} }')

In [ ]:
# Folium map with clustered markers coloured by category
m = folium.Map(location=[20, 0], zoom_start=2, tiles='CartoDB dark_matter')
marker_cluster = MarkerCluster(name='Anchors').add_to(m)

for anchor in anchors:
    folium.CircleMarker(
        location=[anchor['lat'], anchor['lng']],
        radius=6,
        color=CAT_COLORS[anchor['category']],
        fill=True,
        fill_opacity=0.8,
        popup=folium.Popup(
            f"<b>{anchor['title']}</b><br>"
            f"Category: {anchor['category']}<br>"
            f"Reactions: {anchor['reaction_count']}<br>"
            f"City: {anchor['city']}",
            max_width=200,
        ),
    ).add_to(marker_cluster)

m.save('spatial_canvas_map.html')
print('Saved to spatial_canvas_map.html')
# Display inline (requires Jupyter with iframe support)
display(IFrame('spatial_canvas_map.html', width=800, height=450))

In [ ]:
# Reaction heatmap overlay
m_heat = folium.Map(location=[20, 0], zoom_start=2, tiles='CartoDB dark_matter')
heat_data = [[a['lat'], a['lng'], a['reaction_count'] / 150.0] for a in anchors]
HeatMap(heat_data, radius=25, blur=15, name='Reaction Heatmap').add_to(m_heat)
m_heat.save('reaction_heatmap.html')
print('Saved to reaction_heatmap.html')
display(IFrame('reaction_heatmap.html', width=800, height=450))

In [ ]:
# Bar chart — anchor density by city
city_counts = {city: 0 for city in city_names}
city_reactions = {city: 0 for city in city_names}
for a in anchors:
    city_counts[a['city']] += 1
    city_reactions[a['city']] += a['reaction_count']

fig_bar = go.Figure()
fig_bar.add_trace(go.Bar(
    x=list(city_counts.keys()),
    y=list(city_counts.values()),
    name='Anchor Count',
    marker_color='rgba(99,102,241,0.8)',
))
fig_bar.add_trace(go.Bar(
    x=list(city_reactions.keys()),
    y=[v // 10 for v in city_reactions.values()],
    name='Reactions ÷ 10',
    marker_color='rgba(16,185,129,0.7)',
))
fig_bar.update_layout(
    title='Anchor Density & Reactions by City',
    barmode='group',
    xaxis_title='City',
    yaxis_title='Count',
    paper_bgcolor='#0a0a0f',
    plot_bgcolor='#0a0a0f',
    font=dict(color='#f8fafc'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.07)'),
    yaxis_gridcolor='rgba(255,255,255,0.07)',
    legend=dict(bgcolor='rgba(255,255,255,0.05)', bordercolor='rgba(255,255,255,0.1)', borderwidth=1),
)
fig_bar.show()

In [ ]:
# Time-series — anchor creation rate over 90 days
from collections import Counter

date_counts = Counter(a['created_at'] for a in anchors)
all_dates = sorted(date_counts.keys())
counts = [date_counts.get(d, 0) for d in all_dates]

# Fill missing dates with 0
full_dates = [base_date + datetime.timedelta(days=i) for i in range(91)]
full_counts = [date_counts.get(d, 0) for d in full_dates]

fig_time = go.Figure()
fig_time.add_trace(go.Scatter(
    x=full_dates,
    y=full_counts,
    mode='lines+markers',
    fill='tozeroy',
    line=dict(color='#6366f1', width=2),
    fillcolor='rgba(99,102,241,0.15)',
    marker=dict(size=4, color='#6366f1'),
    name='Anchors Created',
))
fig_time.update_layout(
    title='Anchor Creation Rate — Last 90 Days',
    xaxis_title='Date',
    yaxis_title='Anchors Created',
    paper_bgcolor='#0a0a0f',
    plot_bgcolor='#0a0a0f',
    font=dict(color='#f8fafc'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.07)'),
    yaxis_gridcolor='rgba(255,255,255,0.07)',
)
fig_time.show()

In [ ]:
# 3D scatter: lat × lng × reaction_count, coloured by category
cat_to_num = {c: i for i, c in enumerate(CATEGORIES)}

fig_3d = go.Figure()
for cat in CATEGORIES:
    cat_anchors = [a for a in anchors if a['category'] == cat]
    if not cat_anchors:
        continue
    fig_3d.add_trace(go.Scatter3d(
        x=[a['lat'] for a in cat_anchors],
        y=[a['lng'] for a in cat_anchors],
        z=[a['reaction_count'] for a in cat_anchors],
        mode='markers',
        marker=dict(
            size=5,
            color=CAT_HEX[cat],
            opacity=0.85,
            line=dict(color='rgba(255,255,255,0.2)', width=0.5),
        ),
        name=cat.capitalize(),
        hovertext=[
            f"<b>{a['title']}</b><br>Category: {a['category']}<br>"
            f"Reactions: {a['reaction_count']}<br>City: {a['city']}"
            for a in cat_anchors
        ],
        hoverinfo='text',
    ))

fig_3d.update_layout(
    title='3D Scatter — Lat × Lng × Reactions',
    scene=dict(
        xaxis_title='Latitude',
        yaxis_title='Longitude',
        zaxis_title='Reaction Count',
        bgcolor='#0a0a0f',
        xaxis=dict(gridcolor='rgba(255,255,255,0.08)'),
        yaxis=dict(gridcolor='rgba(255,255,255,0.08)'),
        zaxis_gridcolor='rgba(255,255,255,0.08)',
    ),
    paper_bgcolor='#0a0a0f',
    font=dict(color='#f8fafc'),
    margin=dict(l=0, r=0, b=0, t=50),
    legend=dict(bgcolor='rgba(255,255,255,0.05)', bordercolor='rgba(255,255,255,0.1)', borderwidth=1),
)
fig_3d.show()

## 🎛️ Sandbox — Interactive Controls

Use the controls below to filter anchors by category, minimum reactions, and recency, then re-render the 3D scatter live.

In [ ]:
output = widgets.Output()

cat_dropdown = widgets.Dropdown(
    options=['all'] + CATEGORIES,
    value='all',
    description='Category:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='350px'),
)
reaction_slider = widgets.IntSlider(
    value=0, min=0, max=150, step=5,
    description='Min Reactions:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='450px'),
)
days_slider = widgets.IntSlider(
    value=90, min=7, max=90, step=7,
    description='Days Back:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='450px'),
)

def update_3d(category, min_reactions, days_back):
    with output:
        output.clear_output(wait=True)
        cutoff = base_date + datetime.timedelta(days=90 - days_back)
        filtered = [
            a for a in anchors
            if (category == 'all' or a['category'] == category)
            and a['reaction_count'] >= min_reactions
            and a['created_at'] >= cutoff
        ]
        print(f'Filtered anchors: {len(filtered)} / {len(anchors)}')

        cats_to_show = CATEGORIES if category == 'all' else [category]
        fig = go.Figure()
        for cat in cats_to_show:
            sub = [a for a in filtered if a['category'] == cat]
            if not sub:
                continue
            fig.add_trace(go.Scatter3d(
                x=[a['lat'] for a in sub],
                y=[a['lng'] for a in sub],
                z=[a['reaction_count'] for a in sub],
                mode='markers',
                marker=dict(size=5, color=CAT_HEX[cat], opacity=0.85),
                name=cat.capitalize(),
                hovertext=[f"{a['title']}<br>Reactions: {a['reaction_count']}" for a in sub],
                hoverinfo='text',
            ))
        fig.update_layout(
            title=f'3D Scatter — Filtered ({len(filtered)} anchors)',
            scene=dict(
                xaxis_title='Latitude', yaxis_title='Longitude', zaxis_title='Reactions',
                bgcolor='#0a0a0f',
            ),
            paper_bgcolor='#0a0a0f',
            font=dict(color='#f8fafc'),
            margin=dict(l=0, r=0, b=0, t=50),
        )
        fig.show()

widgets.interact(update_3d, category=cat_dropdown,
                 min_reactions=reaction_slider, days_back=days_slider)
display(output)